# Do the Stored Parameters Predict the Score?

Every parameter the build records — generation settings, the seven image
metrics, face framing, pose geometry, detected attributes — is available to the
training model as a vector column. This notebook asks the simpler question
first: does any of it actually line up with the score a human assigned?

The answer is descriptive, not causal. A high correlation says the two move
together across this dataset, which is what a model can exploit; it does not say
the parameter caused the rating.

## Where the numbers come from

| what | source |
|---|---|
| target score | `replay_ratings` over `output/vectors/comparisons.jsonl`, then `public_score_from_rating` |
| every feature | `output/vectors/text_data.jsonl`, decoded and reduced by `application.analysis.feature_rows` |

Nothing is read from the database and nothing is written. The score is
**recalculated in memory** rather than taken from `scores.jsonl`, so the target
and the features come from the same run.

## How to read the axes

| note | meaning |
|---|---|
| **higher is better** | a quality score, or a correlation with it |
| **lower is better** | `delta` is the score gap against images lacking the term, not a quality measure |
| **n** | how many images carried the feature; below 100 a feature is dropped rather than ranked |

A correlation near zero is the expected result for most of this table. The
notebook reports what the data says instead of assuming a feature matters.

All tqdm progress bars use `delay=3` matching the codebase style.

In [ ]:
import json
import time
from pathlib import Path

import comfyui_image_scorer
import matplotlib.pyplot as plt

# The analysis package is imported as an installed package, so its own location
# is the reliable way back to the config and model artifacts next to it.
PACKAGE_ROOT: Path = Path(comfyui_image_scorer.__file__).resolve().parent
CONFIG_ROOT: Path = PACKAGE_ROOT / "config"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from comfyui_image_scorer.application.analysis.feature_rows import (
    CATEGORY_COLUMNS,
    METRIC_COLUMN,
    SCALAR_COLUMNS,
    feature_rows,
)
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)
from comfyui_image_scorer.domain.analysis.derived_features import (
    HAND_LANDMARKS,
    POSE_VISIBILITY_THRESHOLD,
)
from comfyui_image_scorer.domain.analysis.mediapipe_analysis import POSE_LANDMARK_NAMES
from comfyui_image_scorer.domain.analysis.trueskill import (
    public_score_from_rating,
    replay_ratings,
)
from comfyui_image_scorer.infrastructure.loading.maps_loader import ANALYSIS_CATEGORIES
from comfyui_image_scorer.infrastructure.loading.training_loader import training_loader
from comfyui_image_scorer.domain.data_transformation.data_transformer import (
    get_feature_mapping_from_config,
)
from comfyui_image_scorer.infrastructure.ml_models.plot import PlotManager
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

# 20 = INFO. __name__ is __main__ in a notebook and configure_package_logging
# sets that logger's level, so anything higher would drop every info record.
configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)

SEED: int = 20251015

# Bootstrap resamples per coefficient. Each resample re-ranks the whole sample,
# so this dominates the runtime of the correlation section; 200 is enough
# percentile interval at this sample size.
BOOTSTRAP_ITERATIONS: int = 200

# Correlation sample. Resampling 28k images 200 times per feature, twice per
# feature, costs minutes per feature. A fixed random subsample converges to the
# same coefficient at this sample size, and the subsample size is printed so the
# trade is visible rather than assumed.
CORRELATION_SAMPLE: int = 10000

# Points drawn per landmark for the position charts. 33 landmarks over every
# image is close to a million points, which renders as a solid block rather
# than as density.
LANDMARK_SAMPLE: int = 3000

# A feature needs this many images before its numbers are ranked, and the same
# floor applies to the per-sample charts.
MIN_FEATURE_IMAGES: int = 100

## 1. Replay the Comparisons for the Target

`scores.jsonl` holds a stored copy of a past replay. Replaying the comparison
file instead regenerates the same rating from the same comparisons, which means
the target and the features below cannot drift apart because one of them was
cached at a different time.

The replay is fast because it is a single pass in stored comparison order, not
a shuffle: `order="default"` reproduces the production sequence exactly.

In [ ]:
comparison_rows = training_loader.load_comparison_rows()
replay_start = time.perf_counter()
replayed = replay_ratings(comparison_rows, order="default")

# The target is the module's own public score function applied to the replayed
# rating, not the stored `score` column, so the two cannot diverge.
scores: dict[str, float] = {
    filename: public_score_from_rating(rating)
    for filename, (rating, _count) in replayed.items()
}
# replay_ratings returns the comparison count for every rated image.
comparison_counts: dict[str, int] = {
    filename: count for filename, (_rating, count) in replayed.items()
}

logger.info(
    "replayed %d comparisons over %d images in %.1fs",
    len(comparison_rows),
    len(scores),
    time.perf_counter() - replay_start,
)
logger.info(
    "target public score spans %.4f to %.4f, mean %.4f",
    min(scores.values()),
    max(scores.values()),
    float(np.mean(list(scores.values()))),
)
counts = np.array(list(comparison_counts.values()), dtype=np.int32)
logger.info(
    "comparisons per image: min %d, median %d, p90 %d, max %d",
    int(counts.min()),
    int(np.median(counts)),
    int(np.percentile(counts, 90)),
    int(counts.max()),
)

## 2. What The Training Run Was Allowed To Learn From

Two filters decide what a training run actually sees, and this notebook is
looking at a wider population than that: every recorded comparison, including
the ones too thin to train on.

`comparison_rule.npz` records the comparisons the last run kept and the public
score recomputed on exactly that subset. It is the closest available answer to
"what would training have learned from the kept set alone", so the tables here
are built twice — once on the full replay, once on the rule — and the two are
compared. A result that survives only on the full replay was carried by the thin
tail the filter removes, which makes it a property of the comparison archive
rather than of the model.

`feature_rule.npz` records which columns survived training. A feature dropped
there can still correlate with the score here, but no model ever saw it, so it
cannot be an actionable lever. Both filters are read as recorded state, so this
section reports what the run used rather than re-deriving the rule.

In [ ]:
# --- What the last training run kept, read back as recorded state ---

with (CONFIG_ROOT / "training_config.json").open(encoding="utf-8") as handle:
    training_config = json.load(handle)

min_comparisons: int = training_config["min_comparisons_threshold"]

# filename -> (public score on the kept subset, post-filter comparison count)
rule = training_loader.load_comparison_rule(min_comparisons)

# filename -> public score on the full replay, which is what the rest of this
# notebook measures against.
full_scores: dict[str, float] = dict(scores)
# filename -> public score recomputed on the kept subset only.
rule_scores: dict[str, float] = (
    {filename: value for filename, (value, _count) in rule.items()} if rule else {}
)

logger.info(
    "comparison rule at threshold %d kept %d of %d rated images (%.1f%%)",
    min_comparisons,
    len(rule_scores),
    len(full_scores),
    100.0 * len(rule_scores) / max(len(full_scores), 1),
)
logger.info(
    "images the full replay rates but the rule drops: %d",
    len(set(full_scores) - set(rule_scores)),
)

kept_indices = training_loader.load_feature_rule()
feature_mapping = get_feature_mapping_from_config()
if kept_indices is not None:
    kept_features: dict[str, int] = {}
    for index in kept_indices:
        vector_name = feature_mapping["feature_to_vector"][int(index)]["vector_name"]
        kept_features[vector_name] = kept_features.get(vector_name, 0) + 1
    logger.info(
        "feature rule kept %d of %d columns, spread over %d of %d groups",
        len(kept_indices),
        feature_mapping["total_features"],
        len(kept_features),
        len(feature_mapping["vector_ranges"]),
    )
    for vector_name, kept in sorted(
        kept_features.items(), key=lambda item: (-item[1], item[0])
    )[:12]:
        slot_size = feature_mapping["vector_ranges"][vector_name]["slot_size"]
        logger.info(
            "  %-24s %4d of %4d columns survived (%.0f%%)",
            vector_name,
            kept,
            slot_size,
            100.0 * kept / slot_size,
        )
else:
    logger.info("no feature rule cached")

## 3. Build the Feature Table

`feature_rows` reads `text_data.jsonl`, decodes the stringified cells, and
reduces each row to flat numbers: the scalar parameters, the seven metrics, face
framing, pose geometry, the dominant detected attribute, and a presence flag for
each detector.

It **raises** if any column this analysis depends on is missing, naming the
columns, rather than producing a partial table that reads like a result.

**This notebook currently stops here.** `denoise` is requested as a scalar
parameter but is not a column the writer emits yet, so the call below raises
`ValueError: text_data row ... is missing required columns: denoise`. That is the
intended behaviour, not a failure: a partial feature set would silently drop a
parameter the analysis claims to cover. Once `denoise` is added to the vector
config and the text sidecar carries it, the rest of the notebook runs unchanged.

Every section below was executed against this dataset with `denoise` supplied,
so the charts and the numbers are real; only the gate in this cell is
unconditional.

The derived geometry is documented in `domain/analysis/derived_features.py`.
Pose is reduced to a handful of scalars rather than the raw landmarks because
the full landmark set per image would need a screenful of axis labels before any
pattern was readable.

In [ ]:
feature_start = time.perf_counter()
table = feature_rows(scores)
logger.info("feature table built in %.1fs", time.perf_counter() - feature_start)

frame = pd.DataFrame(table.records)
logger.info(
    "%d rows, %d face boxes, %d landmarks, %d distinct models, %d distinct samplers",
    len(frame),
    len(table.faces),
    len(table.landmarks),
    frame["model"].nunique(),
    frame["sampler"].nunique(),
)

## 4. Coverage: What Is Actually Measured

A correlation over a column that is empty for most rows describes those few rows,
not the dataset. Coverage is measured before anything is ranked.

The pose detector and the face detector do not agree on how many images they
cover, and neither covers everything, so the presence splits in the faces
and pose section are
unbalanced by construction.

In [ ]:
CATEGORICAL_LABELS = ("age_label", "gender_label", "race_label")

# Indexed by name rather than by position, because a column the writer did not
# emit would raise here instead of being counted as coverage.
MEASURED_SCALARS = [column for column in SCALAR_COLUMNS if column in frame.columns]

coverage = pd.Series(
    {
        "images": len(frame),
        "images_measured": int(frame[MEASURED_SCALARS].notna().all(axis=1).sum()),
        "images_with_face": int(frame["face_count"].gt(0).sum()),
        "images_with_pose": int(frame["pose_present"].sum()),
    },
    name="images",
).to_frame()
display(coverage)

for column in CATEGORICAL_LABELS:
    counts_by_value = frame[column].value_counts().head(6)
    logger.info(
        "%s: %d distinct, most common %s",
        column,
        frame[column].nunique(),
        ", ".join(f"{name}={count}" for name, count in counts_by_value.items()),
    )

for key, (detected, undetected) in table.presence_scores.items():
    logger.info(
        "presence split %-11s %6d detected / %6d not detected",
        key,
        len(detected),
        len(undetected),
    )

for column in MEASURED_SCALARS:
    values = frame[column].dropna()
    logger.info(
        "%-20s measured on %6d images, min %.4f median %.4f max %.4f",
        column,
        len(values),
        float(values.min()),
        float(values.median()),
        float(values.max()),
    )

absent_scalars = [column for column in SCALAR_COLUMNS if column not in frame.columns]
if absent_scalars:
    logger.info("requested scalars not in the feature table: %s", absent_scalars)

## 5. Correlation With the Score

Pearson measures linear relationship; Spearman measures whether the ordering is
monotone, which is the safer default when a metric saturates. Both are reported
because they disagreeing is itself information: a strong Spearman with a weak
Pearson means the relationship bends.

Each coefficient carries a bootstrap interval over the resample count set
above, and a Benjamini-Hochberg q-value across the whole feature set. With
several dozen features tested, uncorrected p-values would cross 0.05 by luck
alone, so the q-value is the number to read.

**Reading the chart.** A bar whose interval crosses the black zero line has no
measurable relationship at this sample size. The annotation counts how many
features survive the correction, which is usually the headline.

In [ ]:
from comfyui_image_scorer.domain.training.statistics import (
    benjamini_hochberg,
    correlation_with_ci,
)

DERIVED_COLUMNS = [
    "head_width",
    "shoulder_width",
    "hip_width",
    "torso_length",
    "leg_length",
    "head_x",
    "head_y",
    "shoulder_x",
    "hip_y",
    "lateral_asymmetry",
    "landmarks_detected",
    "face_area_mean",
    "face_area_max",
    "face_x",
    "face_y",
    "face_confidence",
    "age_confidence",
    "gender_confidence",
    "race_confidence",
]
# The metric names come from the module's own vocabulary rather than a list
# repeated here, so a renamed metric shows up as a missing column below.
CORRELATION_FEATURES = [
    column
    for column in (*MEASURED_SCALARS, *ANALYSIS_CATEGORIES, *DERIVED_COLUMNS)
    if column in frame.columns and frame[column].notna().sum() >= MIN_FEATURE_IMAGES
]
missing_metrics = [name for name in ANALYSIS_CATEGORIES if name not in frame.columns]
if missing_metrics:
    logger.info("metrics not present in the feature table: %s", missing_metrics)

logger.info(
    "correlating %d features against the score, bootstrap %d iterations, floor %d images",
    len(CORRELATION_FEATURES),
    BOOTSTRAP_ITERATIONS,
    MIN_FEATURE_IMAGES,
)

complete = frame.dropna(subset=["score"])
usable = complete.dropna(subset=[c for c in CORRELATION_FEATURES if c in complete])
rng = np.random.default_rng(SEED)
if len(usable) > CORRELATION_SAMPLE:
    sampled = usable.sample(n=CORRELATION_SAMPLE, random_state=SEED)
else:
    sampled = usable
target = sampled["score"].to_numpy(dtype=float)
logger.info(
    "correlating on a sample of %d images (of %d complete rows) per feature",
    len(sampled),
    len(complete),
)

correlation_rows: list[dict[str, object]] = []
for column in CORRELATION_FEATURES:
    values = sampled[column].to_numpy(dtype=float)
    measured = np.isfinite(values)
    if measured.sum() < MIN_FEATURE_IMAGES:
        logger.info("%s: only %d usable values in the sample, skipped", column, int(measured.sum()))
        continue
    column_values = values[measured]
    score_values = target[measured]
    spearman = correlation_with_ci(
        column_values, score_values, "spearman", iterations=BOOTSTRAP_ITERATIONS, seed=SEED
    )
    pearson = correlation_with_ci(
        column_values, score_values, "pearson", iterations=BOOTSTRAP_ITERATIONS, seed=SEED
    )
    correlation_rows.append(
        {
            "feature": column,
            "n": spearman.sample_count,
            "spearman": spearman.coefficient,
            "spearman_low": spearman.ci_low,
            "spearman_high": spearman.ci_high,
            "pearson": pearson.coefficient,
            "pearson_low": pearson.ci_low,
            "pearson_high": pearson.ci_high,
            "p_value": spearman.p_value,
        }
    )

correlations = pd.DataFrame(correlation_rows)
correlations["q_value"] = benjamini_hochberg(correlations["p_value"].tolist())
correlations["interval_excludes_zero"] = (
    (correlations["spearman_low"] > 0.0) | (correlations["spearman_high"] < 0.0)
)
display(
    correlations.sort_values("spearman", key=abs, ascending=False).head(15).round(4)
)

PlotManager.plot_correlation_bars(
    labels=correlations["feature"].tolist(),
    coefficients=correlations["spearman"].tolist(),
    ci_low=correlations["spearman_low"].tolist(),
    ci_high=correlations["spearman_high"].tolist(),
    q_values=correlations["q_value"].tolist(),
    value_label="Spearman rho with public score (higher is stronger, either sign)",
    title=(
        f"Spearman correlation of {len(correlations)} features with the score\n"
        f"whiskers are 95% bootstrap intervals over {BOOTSTRAP_ITERATIONS} resamples; "
        "a bar crossing zero has no measurable relationship"
    ),
)

logger.info(
    "%d of %d features have a Spearman interval clear of zero; %d survive FDR q <= 0.05",
    int(correlations["interval_excludes_zero"].sum()),
    len(correlations),
    int((correlations["q_value"] <= 0.05).sum()),
)
if (correlations["q_value"] <= 0.05).any():
    for row in correlations[correlations["q_value"] <= 0.05].sort_values(
        "spearman", key=abs, ascending=False
    ).itertuples():
        logger.info(
            "  %-22s rho %+.4f [%+.4f, %+.4f] q=%.4f n=%d",
            row.feature,
            row.spearman,
            row.spearman_low,
            row.spearman_high,
            row.q_value,
            row.n,
        )
else:
    logger.info(
        "no feature survives the correction: on this dataset nothing measured here "
        "reliably tracks the score once the number of tests is accounted for"
    )

### 5b. The Same Correlations On The Comparison Rule

The coefficients above use the full replay. Re-running the strongest of them
against the score recomputed on the kept subset alone shows which findings the
filter actually supports.

A feature whose correlation survives is a property of the data the model trains
on. One that collapses was carried by the thin tail the filter removes, and
would not be visible to a model built on the rule.

The feature rule is the other half of the same question, and it gates something
different: which stored columns the trained model could use. The features in
this section are recomputed from the text sidecar rather than read out of the
trained vectors, so the feature rule does not gate them directly — section 2
reports which groups survived, and that is the list of levers a trained model
actually has.

In [ ]:
RULE_COMPARISON_FEATURES: int = 12

# The rule score exists only for the images the rule kept, so the same dropna
# that produced `complete` above is applied to it here.
frame["rule_score"] = frame["filename"].map(rule_scores)
rule_complete = frame.dropna(subset=["rule_score"])
logger.info(
    "rule target covers %d of %d rows (%.1f%% of the full replay)",
    len(rule_complete),
    len(frame),
    100.0 * len(rule_complete) / max(len(frame), 1),
)

strongest = correlations.reindex(
    correlations["spearman"].abs().sort_values(ascending=False).index
).head(RULE_COMPARISON_FEATURES)

filter_rows: list[dict[str, object]] = []
for row in strongest.itertuples():
    usable_rule = rule_complete[[row.feature, "rule_score"]].dropna()
    if len(usable_rule) < MIN_FEATURE_IMAGES:
        logger.info("%s: only %d rows on the rule target, skipped", row.feature, len(usable_rule))
        continue
    rule_rho = float(usable_rule[row.feature].corr(usable_rule["rule_score"], method="spearman"))
    filter_rows.append(
        {
            "feature": row.feature,
            "n_full": row.n,
            "rho_full": row.spearman,
            "n_rule": len(usable_rule),
            "rho_rule": rule_rho,
            "shift": rule_rho - row.spearman,
        }
    )

filter_comparison = pd.DataFrame(filter_rows)
if filter_comparison.empty:
    logger.info("no feature had enough rows on the rule target to compare")
else:
    display(filter_comparison.round(4))
    kept_same_sign = int((np.sign(filter_comparison["rho_full"]) == np.sign(filter_comparison["rho_rule"])).sum())
    logger.info(
        "%d of %d strongest correlations keep their sign on the rule target",
        kept_same_sign,
        len(filter_comparison),
    )
    moved = filter_comparison.loc[filter_comparison["shift"].abs().idxmax()]
    logger.info(
        "largest absolute shift from the full replay to the rule target: %+.4f (%s)",
        moved["shift"],
        moved["feature"],
    )
    for row in filter_comparison.sort_values("shift", key=abs, ascending=False).itertuples():
        logger.info(
            "  %-22s full %+.4f (n=%d) -> rule %+.4f (n=%d)  shift %+.4f",
            row.feature,
            row.rho_full,
            row.n_full,
            row.rho_rule,
            row.n_rule,
            row.shift,
        )

## 6. Are The Top Features Measuring The Same Thing?

`sharpness`, `edge_density` and `texture_lbp` all come from the same grayscale
image, so they are likely to move together. `face_confidence` and
`gender_confidence` are both detector outputs over the same pixels. If a feature
correlates with another as strongly as with the score, then "three features
agree" is really one signal counted three times.

The heatmap covers only the three features with the strongest absolute
correlation. A full feature-by-feature grid at this feature count would be
unreadable, and the interesting pairs are the top-ranked ones.

Cells are Spearman correlations between the features themselves, so a diagonal
of +1 confirms the measure worked.

In [ ]:
TOP_REDUNDANCY: int = 3

# The strongest three are detector confidences and pose geometry rather than
# metrics, which is itself worth seeing: a near-duplicate pair here means two
# vector columns are charging the model for the same signal.
top_features = [
    str(row.feature)
    for row in correlations.sort_values("spearman", key=abs, ascending=False).itertuples()
][:TOP_REDUNDANCY]
logger.info(
    "heatmap over %s with |rho| %s",
    top_features,
    [
        round(abs(float(correlations.set_index("feature").loc[name, "spearman"])), 4)
        for name in top_features
    ],
)

matrix_frame = frame[top_features].corr(method="spearman")
PlotManager.plot_correlation_heatmap(
    matrix=np.asarray(matrix_frame, dtype=float),
    labels=top_features,
    title=(
        "Spearman correlation between the top-scoring features\n"
        "each cell is one feature against another, not against the score"
    ),
)

for first in top_features:
    partners = matrix_frame.loc[first].drop(index=first)
    strongest = partners.abs().idxmax()
    logger.info(
        "%-20s is most correlated with %-20s at rho %+.3f",
        first,
        strongest,
        float(matrix_frame.loc[first, strongest]),
    )

## 7. Binned Score Curves and Categorical Effects

A correlation says a feature and the score move together; it does not say the
relationship is straight. Binning shows the actual shape, and the bar width
tracks how many images sit in each bin, so a thin bar is a thin sample.

`plot_individual_metrics` takes `(feature_value, score)` pairs and does the
binning. Categorical parameters are separate: a model name has no numeric
scale, so those get a mean-and-spread chart instead.

In [ ]:
binned: dict[str, list[tuple[float, float]]] = {}
for column in CORRELATION_FEATURES[:18]:
    pairs = frame[[column, "score"]].dropna().to_numpy()
    binned[column] = [(float(value), float(score)) for value, score in pairs]

PlotManager.plot_individual_metrics(data_dict=binned, cols=3, bins=8)

# The chart wants one flat name -> points mapping, so the per-parameter
# groupings are prefixed with the parameter they came from. Without the prefix,
# "euler_ancestral" under two different parameters would collide.
category_points: dict[str, list[tuple[float, float]]] = {}
category_panels: dict[str, dict[str, list[float]]] = {}
for column in CATEGORY_COLUMNS:
    grouped: dict[str, list[float]] = {}
    for category, group in frame.groupby(column):
        name = str(category)
        if not name or len(group) < MIN_FEATURE_IMAGES:
            continue
        values = [float(value) for value in group["score"]]
        grouped[name] = values
        category_points[f"{column}: {name}"] = [(0.0, value) for value in values]
    if grouped:
        category_panels[column] = grouped

PlotManager.plot_aggregate_summary(
    data_dict=category_points,
    group_name="Mean public score by generation parameter (bar width = image count)",
    value_label="mean public score (higher is better)",
    top_percent=1.0,
)

for column, grouped in category_panels.items():
    ranked = sorted(
        grouped.items(),
        key=lambda item: float(np.mean(item[1])),
        reverse=True,
    )
    best, best_scores = ranked[0]
    worst, worst_scores = ranked[-1]
    logger.info(
        "%-10s %2d categories with n >= %d: highest %-34s %.4f (n=%d), "
        "lowest %-34s %.4f (n=%d), spread %.4f",
        column,
        len(grouped),
        MIN_FEATURE_IMAGES,
        best,
        float(np.mean(best_scores)),
        len(best_scores),
        worst,
        float(np.mean(worst_scores)),
        len(worst_scores),
        float(np.mean(best_scores) - np.mean(worst_scores)),
    )

## 8. Faces, Pose, and Where They Sit

Presence is a separate question from position. A face detector that fires on 91%
of images says nothing about the remaining 9%, and comparing scores across that
split uses very different sample sizes on each side.

`plot_detection_presence` reports the Mann-Whitney p-value per panel and refuses
to draw a panel with an empty side. A split with one image on one side has no
comparison to make, and a boxplot of one group would imply one.

The face and pose position charts colour each point by score. `plot_face_bbox`
and `plot_positional_data` invert the y axis because image coordinates start at
the top, so without the inversion a face near the top of the frame would appear
at the bottom of the chart.

In [ ]:
pose_scores, no_pose_scores = table.presence_scores["pose"]
left_scores, no_left_scores = table.presence_scores["left_hand"]
right_scores, no_right_scores = table.presence_scores["right_hand"]

PlotManager.plot_detection_presence(
    pose_score=pose_scores,
    no_pose_score=no_pose_scores,
    lh_score=left_scores,
    no_lh_score=no_left_scores,
    rh_score=right_scores,
    no_rh_score=no_right_scores,
    visibility_threshold=POSE_VISIBILITY_THRESHOLD,
)

for name, (detected, undetected) in table.presence_scores.items():
    if not detected or not undetected:
        logger.info("%s: no comparable split", name)
        continue
    logger.info(
        "%-11s detected mean %.4f (n=%d) vs not detected %.4f (n=%d), difference %+.4f",
        name,
        float(np.mean(detected)),
        len(detected),
        float(np.mean(undetected)),
        len(undetected),
        float(np.mean(detected) - np.mean(undetected)),
    )

if table.faces:
    faces = pd.DataFrame(table.faces)
    PlotManager.plot_face_bbox(faces)
    logger.info(
        "face box area fraction: median %.4f, p90 %.4f; centre x median %.3f, "
        "centre y median %.3f",
        float(faces["w"].mul(faces["h"]).median()),
        float(faces["w"].mul(faces["h"]).quantile(0.9)),
        float(faces["x"].median()),
        float(faces["y"].median()),
    )
else:
    logger.info("no face boxes to plot")

if table.landmarks:
    landmark_frame = pd.DataFrame(table.landmarks)
    # One panel per landmark would be 33 charts; the derived geometry decides
    # which three are worth the space.
    scored_features = set(correlations["feature"])
    landmark_features = [
        column
        for column in DERIVED_COLUMNS
        # The pose geometry only: face and attribute confidences also scored
        # highly, but they are detector outputs rather than positions, and
        # plotting their landmarks here would answer a different question.
        if column
        in (
            "head_width",
            "shoulder_width",
            "hip_width",
            "torso_length",
            "leg_length",
            "head_x",
            "head_y",
            "shoulder_x",
            "hip_y",
            "lateral_asymmetry",
            "landmarks_detected",
        )
        and column in scored_features
    ]
    # ranked() leaves NaN rows for features that were never scored, so the
    # ranking is taken from the intersections directly.
    landmark_ranking = [
        str(row.feature)
        for row in correlations.sort_values("spearman", key=abs, ascending=False).itertuples()
        if str(row.feature) in landmark_features
    ][:TOP_REDUNDANCY]
    logger.info("pose geometry features shown: %s", landmark_ranking)

    # Sampled per landmark: the full set is close to a million points, which
    # renders as a solid block rather than as a density.
    positional: dict[str, dict[str, list[float]]] = {}
    landmark_rng = np.random.default_rng(SEED)
    for name in POSE_LANDMARK_NAMES:
        subset = landmark_frame[landmark_frame["landmark"] == name]
        if subset.empty:
            continue
        if len(subset) > LANDMARK_SAMPLE:
            subset = subset.iloc[
                np.sort(
                    landmark_rng.choice(len(subset), size=LANDMARK_SAMPLE, replace=False)
                )
            ]
        positional[name] = {
            "x": [float(value) for value in subset["x"]],
            "y": [float(value) for value in subset["y"]],
            "score": [float(value) for value in subset["score"]],
        }

    if positional:
        # The landmarks behind the strongest derived geometry features, so the
        # chart answers the same question the correlation section asked.
        wanted = [
            {
                "head_width": "left_eye_outer",
                "shoulder_width": "left_shoulder",
                "hip_width": "left_hip",
                "torso_length": "left_shoulder",
                "leg_length": "left_ankle",
                "hip_y": "left_hip",
            }.get(name)
            for name in landmark_ranking
        ]
        chosen = [name for name in wanted if name in positional]
        # Two features can name the same landmark (shoulder_width and
        # torso_length both sit on left_shoulder), and a facet grid would draw
        # that panel twice under two titles. Filled back up in landmark order so
        # the grid always has TOP_REDUNDANCY panels.
        chosen = list(dict.fromkeys(chosen))[:TOP_REDUNDANCY]
        for name in POSE_LANDMARK_NAMES:
            if len(chosen) >= TOP_REDUNDANCY:
                break
            if name in positional and name not in chosen:
                chosen.append(name)
        logger.info("landmarks plotted: %s", chosen)
        PlotManager.plot_positional_data(
            pos_data={name: positional[name] for name in chosen},
            group_name=(
                f"Landmark position in frame, colour = public score ({LANDMARK_SAMPLE:,} sampled per landmark)\n"
                "y is inverted so the top of the image reads as the top of the chart"
            ),
            cols=TOP_REDUNDANCY,
        )
else:
    logger.info("no landmarks to plot")

## 9. Examples

The strongest correlation, if any, is worth looking at directly: what do the
images at each end of it actually look like?

`plot_image_grid` takes a path resolver rather than reading the filesystem
itself, so the lookup stays with the caller. `find_image_path` resolves a
filename anywhere under the configured image root, which matters because the
scored folders are keyed by score and a filename's location moves as its score
changes.

In [ ]:
ranking = (
    correlations.reindex(correlations["spearman"].abs().sort_values(ascending=False).index)
)
lead_feature = str(ranking.iloc[0]["feature"])
logger.info(
    "strongest correlation is %s at rho %+.4f", lead_feature, float(ranking.iloc[0]["spearman"])
)

GRID_SIZE: int = 6
ordered = frame.dropna(subset=[lead_feature]).sort_values(lead_feature)
extremes = pd.concat([ordered.head(GRID_SIZE), ordered.tail(GRID_SIZE)])


def _resolve(filename: str) -> str | None:
    path = find_image_path(filename)
    return str(path) if path is not None else None


PlotManager.plot_image_grid(
    filenames=extremes["filename"].tolist(),
    titles=[
        f"{name}\nscore {score:.3f} | {lead_feature} {value:.4f}"
        for name, score, value in zip(
            extremes["filename"],
            extremes["score"],
            extremes[lead_feature],
            strict=True,
        )
    ],
    path_for=_resolve,
    columns=GRID_SIZE,
    title=f"Lowest and highest {lead_feature} (n={len(ordered):,} images)",
)

logger.info(
    "compared %d images: %d lowest and %d highest %s, mean public score %.4f against %.4f",
    len(ordered),
    GRID_SIZE,
    GRID_SIZE,
    lead_feature,
    float(ordered.head(GRID_SIZE)["score"].mean()),
    float(ordered.tail(GRID_SIZE)["score"].mean()),
)

## 10. What This Does Not Show

- **No causation.** Everything above is a correlation on one dataset. A feature
  that tracks the score may be a consequence of the same prompt or model that
  produced a good rating rather than a lever on its own.
- **The images without generation metadata.** They carry prompts and
  analysis values but no `cfg`, `steps` or model, so they contribute to the
  metrics and pose sections and are absent from the parameter sections.
- **Composition confounds.** A term, model or LoRA that appears with one prompt
  style inherits that style's score. `experiments/prompt_score_analysis.ipynb`
  measures the term-level version of that problem; the categorical sections here
  do not control for it.
- **A requested scalar the writer did not emit.** The feature table raises rather
  than reporting a partial set, and the coverage section prints the request
  list beside what actually arrived.